<a href="https://colab.research.google.com/github/Harshitha82/Medical-Report-Summarization/blob/main/Medical_report_summarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install google-generativeai
!pip install transformers
!pip install torch
!pip install pdfplumber
!pip install textstat
!pip install accelerate

In [ ]:
import pdfplumber
import google.generativeai as genai
import textstat
from transformers import pipeline
from google.colab import files

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

print(file_name)

Saving Sample-Smart-Report-Clinics.pdf to Sample-Smart-Report-Clinics.pdf
Sample-Smart-Report-Clinics.pdf


In [ ]:
report_text = ""

with pdfplumber.open(file_name) as pdf:

    for page in pdf.pages:

        text = page.extract_text()

        if text:
            report_text += text + "\n"

print(report_text[:3000])

Health Report
Patient Name (Your name) : NI BHASKARAN
Age/Gender (Your age/gender) : 58Y/Female
Apollo Clinic Phone No: 1860 500 7788
#1, Near Mc.Donalds, Prakasam Salai, Valasaravakkam, Chennai, Tamil Nadu, India - 600087 email: kothandaram.rj@apolloclinic.com
Disclaimer:All lab results are subject to clinical interpretation by qualified medical professionals and the report is not subject to use for any medicolegal purpose
Dear RAMANI BHASKARAN (Your name)
Thank you for choosing Apollo ProHealth, India's first AI-powered health management program, curated to help you make positive health shifts. Being healthy is
about making smart choices, and you have taken the first step with this program. We are privileged to be your healthcare partner. Your health is our priority.
We are with you on your path to wellness by:
Predicting your risk: Artificial Intelligence-powered predictive risk scores are generated, based on your personal, medical, and family history and detailed
multi-organ evalua

In [ ]:
ner = pipeline(
    "token-classification",
    model="Clinical-AI-Apollo/Medical-NER",
    aggregation_strategy="simple"
)

entities = ner(report_text)

config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/736M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

In [ ]:
for entity in entities[:50]:
    print(entity)

{'entity_group': 'MEDICATION', 'score': np.float32(0.02601575), 'word': 'µ', 'start': 8019, 'end': 8020}
{'entity_group': 'MEDICATION', 'score': np.float32(0.02583404), 'word': 'µ', 'start': 8038, 'end': 8039}
{'entity_group': 'MEDICATION', 'score': np.float32(0.025772471), 'word': 'µ', 'start': 8097, 'end': 8098}
{'entity_group': 'MEDICATION', 'score': np.float32(0.025504172), 'word': 'µ', 'start': 8115, 'end': 8116}
{'entity_group': 'SEVERITY', 'score': np.float32(0.025135765), 'word': '…', 'start': 15914, 'end': 15915}
{'entity_group': 'DETAILED_DESCRIPTION', 'score': np.float32(0.2508344), 'word': 'Cardiac', 'start': 16803, 'end': 16811}
{'entity_group': 'DISEASE_DISORDER', 'score': np.float32(0.6116434), 'word': 'Coronary Artery Disease', 'start': 16947, 'end': 16971}
{'entity_group': 'DATE', 'score': np.float32(0.2499713), 'word': 'ten years', 'start': 17004, 'end': 17014}
{'entity_group': 'DIAGNOSTIC_PROCEDURE', 'score': np.float32(0.40652308), 'word': 'risk', 'start': 17133, 'e

In [ ]:
medical_data = {
    "diseases": [],
    "medications": [],
    "symptoms": [],
    "procedures": [],
    "lab_tests": [],
    "other": []
}

for entity in entities:

    label = entity["entity_group"]
    word = entity["word"]

    medical_data["other"].append(
        {
            "entity": word,
            "type": label
        }
    )

medical_data

{'diseases': [],
 'medications': [],
 'symptoms': [],
 'procedures': [],
 'lab_tests': [],
 'other': [{'entity': 'µ', 'type': 'MEDICATION'},
  {'entity': 'µ', 'type': 'MEDICATION'},
  {'entity': 'µ', 'type': 'MEDICATION'},
  {'entity': 'µ', 'type': 'MEDICATION'},
  {'entity': '…', 'type': 'SEVERITY'},
  {'entity': 'Cardiac', 'type': 'DETAILED_DESCRIPTION'},
  {'entity': 'Coronary Artery Disease', 'type': 'DISEASE_DISORDER'},
  {'entity': 'ten years', 'type': 'DATE'},
  {'entity': 'risk', 'type': 'DIAGNOSTIC_PROCEDURE'},
  {'entity': 'physical parameters', 'type': 'DIAGNOSTIC_PROCEDURE'},
  {'entity': 'heart health attributes', 'type': 'DIAGNOSTIC_PROCEDURE'},
  {'entity': 'Pre-', 'type': 'DIAGNOSTIC_PROCEDURE'},
  {'entity': 'Diabet', 'type': 'DISEASE_DISORDER'},
  {'entity': 'es', 'type': 'DIAGNOSTIC_PROCEDURE'},
  {'entity': 'Low', 'type': 'LAB_VALUE'},
  {'entity': 'Pre', 'type': 'DIAGNOSTIC_PROCEDURE'},
  {'entity': 'diabetes', 'type': 'DISEASE_DISORDER'},
  {'entity': '2', 'type':

In [ ]:
genai.configure(
    api_key="api key"
)

model = genai.GenerativeModel(
    "gemini-2.5-flash"
)

In [ ]:
analysis_prompt = f"""
Analyze the following medical report.

Return ONLY valid JSON.

Format:

{{
  "lab_findings": [
    {{
      "test": "",
      "value": "",
      "reference_range": "",
      "status": ""
    }}
  ],

  "diseases": [],

  "medications": [],

  "symptoms": [],

  "procedures": []
}}

Medical Report:

{report_text}
"""

In [ ]:
analysis_response = model.generate_content(
    analysis_prompt
)

structured_analysis = analysis_response.text

print(structured_analysis)

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2601.03ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1827.93ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1802.09ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3511.43ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 45060.01ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2134.56ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1831.75ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encodi

```json
{
  "lab_findings": [
    {
      "test": "PCV",
      "value": "35.90",
      "reference_range": "36 - 46 %",
      "status": "Low"
    },
    {
      "test": "HBA1C, GLYCATED HEMOGLOBIN",
      "value": "6.2",
      "reference_range": "0 - 0 %",
      "status": "High"
    },
    {
      "test": "ESTIMATED AVERAGE GLUCOSE (eAG)",
      "value": "131",
      "reference_range": "0 - 0 mg/dL",
      "status": "High"
    },
    {
      "test": "TRIGLYCERIDES",
      "value": "199",
      "reference_range": "0 - 149.99 mg/dL",
      "status": "High"
    },
    {
      "test": "VLDL CHOLESTEROL",
      "value": "39.8",
      "reference_range": "5 - 30 mg/dL",
      "status": "High"
    },
    {
      "test": "UREA",
      "value": "16.00",
      "reference_range": "17 - 43 mg/dL",
      "status": "Low"
    },
    {
      "test": "BLOOD UREA NITROGEN",
      "value": "7.5",
      "reference_range": "8 - 23 mg/dL",
      "status": "Low"
    },
    {
      "test": "VITAMIN B12",
      

In [ ]:
summary_prompt = f"""
Generate a patient-friendly report.

Use EXACTLY this format.

================================================

MEDICAL REPORT SUMMARY

================================================

LAB FINDINGS

- Test:
- Value:
- Status:
- Explanation:

================================================

DISEASES DETECTED

- Disease:
- Explanation:

================================================

MEDICATIONS

- Medication:
- Purpose:

================================================

SYMPTOMS

- Symptom:
- Meaning:

================================================

OVERALL SUMMARY

Provide a short overall explanation.

================================================

DISCLAIMER

This report is for informational purposes only.
Consult a qualified doctor.

================================================

Data:

{structured_analysis}

Do not skip sections.
If no data exists write:

'No significant findings detected.'
"""

In [ ]:
summary_response = model.generate_content(
    summary_prompt
)

summary = summary_response.text

print(summary)


MEDICAL REPORT SUMMARY


LAB FINDINGS

- Test: PCV
- Value: 35.90 %
- Status: Low
- Explanation: Your red blood cell volume is slightly lower than normal, which can sometimes be a sign of mild anemia.

- Test: HBA1C, GLYCATED HEMOGLOBIN
- Value: 6.2 %
- Status: High
- Explanation: This test measures your average blood sugar levels over the past 2-3 months. Your level is slightly high, indicating a risk of developing diabetes (pre-diabetes).

- Test: ESTIMATED AVERAGE GLUCOSE (eAG)
- Value: 131 mg/dL
- Status: High
- Explanation: This value estimates your average daily blood sugar level and is slightly elevated, consistent with the HbA1c finding.

- Test: TRIGLYCERIDES
- Value: 199 mg/dL
- Status: High
- Explanation: Triglycerides are a type of fat in your blood. High levels can increase your risk of heart disease.

- Test: VLDL CHOLESTEROL
- Value: 39.8 mg/dL
- Status: High
- Explanation: VLDL is a type of 'bad' cholesterol. High levels can contribute to hardening of the arteries and 

In [ ]:
print(
    "Reading Ease:",
    textstat.flesch_reading_ease(summary)
)

print(
    "Grade Level:",
    textstat.flesch_kincaid_grade(summary)
)

print(
    "SMOG:",
    textstat.smog_index(summary)
)

Reading Ease: 42.410706262823
Grade Level: 10.239987326757156
SMOG: 11.65658262970966


In [ ]:
!pip install reportlab
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

pdf_file = "medical_summary.pdf"

doc = SimpleDocTemplate(pdf_file)

styles = getSampleStyleSheet()

content = []

content.append(
    Paragraph("Medical Report Summary", styles['Title'])
)

content.append(Spacer(1, 12))

for line in summary.split("\n"):
    if line.strip():
        content.append(
            Paragraph(line, styles['BodyText'])
        )

        content.append(
            Spacer(1, 4)
        )

doc.build(content)

print("PDF Generated Successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.8 MB/s eta 0:00:00
PDF Generated Successfully!


In [ ]:
from google.colab import files

files.download("medical_summary.pdf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>